# Exercícios · Python para Mapeamento Urbano
**Complete o que falta** · Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/))

Substitua cada `??` pela informação que falta e rode a célula com `Shift + Enter`. Se aparecer `SyntaxError`, ainda há um `??` na célula. As dicas estão no texto antes de cada bloco.

| Exercício | Tema | Você vai |
|---|---|---|
| 1 | Favelas e comunidades urbanas | criar colunas, filtrar e agrupar; mapa de densidade; gráfico das maiores favelas |
| 2 | Áreas verdes e densidade | calcular área, filtrar parques; mapa verde sobre densidade; gráfico dos maiores parques |
| 3 | Cheios e vazios | escolher um lugar de BH; mapa figura-fundo; histograma do tamanho das edificações |

Cada exercício termina com um **mapa** e um **gráfico** com os seus créditos, salvos em `outputs/`.

## 0. Ambiente (rode sem mudar nada)

In [ ]:
!pip install -q geobr osmnx mapclassify folium
import os, sys
REPO = "https://github.com/brfelipealmeida/oficina-python-mapeamento"
if not os.path.exists("oficina-python-mapeamento"):
    !git clone -q {REPO}
sys.path.append("oficina-python-mapeamento/src")

import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from estilo import *
from dados import *
configurar_estilo()

## Exercício 1 · Favelas e comunidades urbanas

O Censo 2022 marca os setores censitários que ficam em **Favelas e Comunidades Urbanas** (coluna `CD_FCU`; o nome da favela está em `NM_FCU`). A função `setores_bh()` já cria a coluna `em_favela` (verdadeiro/falso).

### 1.1 Carregar os setores de BH (pronto)

In [ ]:
setores = setores_bh()
setores[["CD_SETOR", "NM_BAIRRO", "NM_FCU", "pessoas", "em_favela"]].head()

### 1.2 Editar o DataFrame

Complete os `??`. Dica: o CRS está em metros, então `setores.area` está em m². Um hectare tem 10.000 m².

In [ ]:
# área em hectares
setores["area_ha"] = setores.area / ??

# densidade: pessoas por hectare
setores["dens_ha"] = setores[??] / setores["area_ha"]

# só os setores em favela
favelas = setores[setores[??] == True]
print(len(favelas), "setores em favelas e comunidades urbanas")

# moradores por favela, das maiores para as menores
por_favela = (favelas.groupby(??)["pessoas"].sum()
              .sort_values(ascending=??)
              .head(10))
por_favela

### 1.3 Mapa

In [ ]:
fig, ax = mapa_base((9, 10))
setores.plot(ax=ax, color=CORES["cinza_claro"], linewidth=0)
favelas.plot(ax=ax, column=??, cmap=CMAP, linewidth=0, legend=True,
             legend_kwds={"shrink": 0.5, "label": "hab/ha"})
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, ??, "Setores censitários, 2022")
creditos(fig, fonte=??)
salvar(fig, "ex1_mapa")

### 1.4 Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
dados = por_favela.sort_values()                 # do menor para o maior: o maior fica no topo
ax.barh(dados.index, dados.values, color=CORES[??])
ax.set_xlabel(??)
titulo(ax, "As dez favelas e comunidades urbanas mais populosas de BH", "Censo 2022")
creditos(fig, fonte="IBGE, Censo 2022")
salvar(fig, "ex1_grafico")

**Para pensar:** a favela mais populosa é também a mais densa? Como você verificaria isso com uma linha de código?

## Exercício 2 · Áreas verdes e densidade

### 2.1 Carregar os parques de BH no OpenStreetMap (pronto)

In [ ]:
verde = osm_feicoes({"leisure": ["park"]}, lugar="Belo Horizonte, Minas Gerais, Brazil")
verde = verde[verde.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
bh = setores                       # setores de BH com densidade, do exercício 1
print(len(verde), "polígonos de parque")

### 2.2 Editar o DataFrame

In [ ]:
# área de cada parque em hectares
verde["area_ha"] = verde.area / ??

# só parques com mais de 1 hectare
grandes = verde[verde["area_ha"] > ??]

# tirar parques sem nome no OSM
grandes = grandes.dropna(subset=[??])

# os dez maiores
top10 = grandes.nlargest(??, "area_ha")[["name", "area_ha"]]
top10

### 2.3 Mapa

In [ ]:
fig, ax = mapa_base((9, 10))
bh.plot(ax=ax, column="dens_ha", scheme="quantiles", k=5, cmap=CMAP, linewidth=0, alpha=0.8)
grandes.plot(ax=ax, color=CORES[??], linewidth=0)
barra_escala(ax, 2000); seta_norte(ax)
titulo(ax, ??, "Parques com mais de 1 ha sobre a densidade por setor")
creditos(fig, fonte="IBGE, Censo 2022; OpenStreetMap")
salvar(fig, "ex2_mapa")

### 2.4 Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
dados = top10.sort_values("area_ha")
ax.barh(dados[??], dados["area_ha"], color=CORES["verde"])
ax.set_xlabel(??)
titulo(ax, "Os dez maiores parques de BH no OpenStreetMap", "Polígonos com tag leisure=park")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "ex2_grafico")

**Para pensar:** o OSM é colaborativo. Algum parque que você conhece ficou de fora? O que isso diz sobre usar só essa fonte?

## Exercício 3 · Cheios e vazios

### 3.1 Escolha um lugar de BH

Copie a latitude e a longitude do Google Maps (clique com o botão direito no mapa). O exemplo é a Praça Sete.

In [ ]:
PONTO = (??, ??)    # (latitude, longitude)
RAIO = ??                        # metros

recorte = gpd.GeoSeries(gpd.points_from_xy([PONTO[1]], [PONTO[0]]), crs=CRS_WEB).to_crs(CRS_BH).buffer(RAIO)
edif = osm_feicoes({"building": True}, ponto=PONTO, raio=RAIO)
edif = edif.clip(recorte).explode(index_parts=False)
edif = edif[edif.geom_type == "Polygon"].copy()
print(len(edif), "edificações")

### 3.2 Editar o DataFrame

In [ ]:
# área de cada edificação em m²
edif["area_m2"] = edif.??

# taxa de ocupação: área construída / área do círculo
taxa = edif["area_m2"].sum() / recorte.area.iloc[0]
print(f"Taxa de ocupação: {taxa:.0%}")

# as cinco maiores edificações
edif.nlargest(??, "area_m2")[["building", "area_m2"]]

### 3.3 Mapa figura-fundo

In [ ]:
fig, ax = mapa_base((9, 9))
edif.plot(ax=ax, color=??, linewidth=0)
recorte.boundary.plot(ax=ax, color=CORES["cinza"], linewidth=0.8)
barra_escala(ax, 200); seta_norte(ax)
titulo(ax, ??, f"Raio de {RAIO} m · {taxa:.0%} da área ocupada")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "ex3_mapa")

### 3.4 Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(edif["area_m2"], bins=??, range=(0, edif["area_m2"].quantile(0.95)), color=CORES["roxo"])
ax.set_xlabel(??); ax.set_ylabel("Quantidade")
titulo(ax, "Tamanho das edificações", "Até o percentil 95, para os extremos não esconderem o resto")
creditos(fig, fonte="OpenStreetMap")
salvar(fig, "ex3_grafico")

**Para pensar:** compare a taxa de ocupação de dois lugares (por exemplo, Praça Sete e um bairro residencial). O que muda no mapa figura-fundo?

---
Elaboração: Felipe Almeida ([felipe-almeida.com](https://felipe-almeida.com/)). Dados: IBGE (Censo 2022), contribuidores do OpenStreetMap.